In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv('rps_data.csv', sep=',')

In [47]:
import pandas as pd
from collections import Counter

# Загрузка данных
df = pd.read_csv('rps_data.csv')

# Условия по раундам
cond1 = (df['round'] == 1) & (df['opp_move'] == 'Б') & (df['outcome'] == 'lose')
cond2 = (df['round'] == 2) & (df['opp_move'] == 'Н') & (df['outcome'] == 'lose')
cond3 = (df['round'] == 3) & (df['opp_move'] == 'Б') & (df['outcome'] == 'draw')

# Находим match_id для каждого условия
matches1 = set(df.loc[cond1, 'match_id'])
matches2 = set(df.loc[cond2, 'match_id'])
matches3 = set(df.loc[cond3, 'match_id'])

# Пересечение
matches = matches1 & matches2 & matches3

print(f"Найдено матчей: {len(matches)}")
print("ID матчей:", sorted(matches))

# Смотрим 4-й раунд
fourth_moves = []
for m in matches:
    round4 = df[(df['match_id'] == m) & (df['round'] == 4)]
    if not round4.empty:
        fourth_moves.append(round4.iloc[0]['opp_move'])

print("\nХоды противника в 4-м раунде:", fourth_moves)
if fourth_moves:
    cnt = Counter(fourth_moves)
    print("\nСтатистика:")
    for move, count in cnt.items():
        print(f"  {move}: {count} раз(а)")
    most_common = cnt.most_common(1)[0]
    print(f"\nНаиболее частый ход: {most_common[0]} (встретился {most_common[1]} раз(а))")
else:
    print("Нет подходящих ситуаций.")

Найдено матчей: 0
ID матчей: []

Ходы противника в 4-м раунде: []
Нет подходящих ситуаций.


In [3]:
df

,match_id,round,player_name,opp_match_wins,opp_match_winrate,stake,opp_move,my_move,outcome,score_me_before,score_opp_before,prev_opp_move,prev_outcome,streak_draws,prev2_opp_move
0,1,1,NaN,0.0,0.0000,25.0,Н,Б,lose,0.0,0,-1,none,0.0,NaN
1,1,2,NaN,0.0,0.0000,25.0,К,Н,lose,0.0,1,Н,lose,0.0,NaN
2,1,3,NaN,0.0,0.0000,25.0,Н,Н,draw,0.0,2,К,lose,0.0,NaN
3,1,4,NaN,0.0,0.0000,25.0,Н,Н,draw,0.0,2,Н,draw,1.0,NaN
4,1,5,NaN,0.0,0.0000,25.0,Н,К,win,0.0,2,Н,draw,2.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1079,170,1,Чувствительный Весельчак,2528.0,0.5952,50.0,Б,К,lose,0.0,0,-1,none,0.0,-1
1080,170,2,Чувствительный Весельчак,2528.0,0.5952,50.0,Н,К,win,0.0,1,Б,lose,0.0,-1
1081,170,3,Чувствительный Весельчак,2528.0,0.5952,50.0,К,Н,lose,1.0,1,Н,win,0.0,Б
1082,170,4,Чувствительный Весельчак,2528.0,0.5952,50.0,К,Б,win,1.0,2,К,lose,0.0,Н


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1084 entries, 0 to 1083
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   match_id           1084 non-null   int64  
 1   round              1084 non-null   int64  
 2   player_name        64 non-null     str    
 3   opp_match_wins     1084 non-null   float64
 4   opp_match_winrate  1084 non-null   float64
 5   stake              1084 non-null   float64
 6   opp_move           1084 non-null   str    
 7   my_move            1084 non-null   str    
 8   outcome            1084 non-null   str    
 9   score_me_before    1084 non-null   float64
 10  score_opp_before   1084 non-null   int64  
 11  prev_opp_move      1084 non-null   str    
 12  prev_outcome       1084 non-null   str    
 13  streak_draws       1084 non-null   float64
 14  prev2_opp_move     926 non-null    str    
dtypes: float64(5), int64(3), str(7)
memory usage: 145.5 KB


In [6]:
class AdaptiveRPSPredictor:
    def __init__(self):
        self.first_move_probs = {'К': 0.33, 'Н': 0.33, 'Б': 0.34}
        self.transitions = defaultdict(Counter)
        self.prev_opp_move = None
        
    def update(self, opp_move):
        if self.prev_opp_move is not None:
            self.transitions[self.prev_opp_move][opp_move] += 1
        self.prev_opp_move = opp_move
        
    def predict_proba(self):
        if self.prev_opp_move is None:
            total = sum(self.first_move_probs.values())
            return {k: v/total for k, v in self.first_move_probs.items()}
        else:
            counter = self.transitions[self.prev_opp_move]
            total = sum(counter.values())
            if total == 0:
                return {'К': 1/3, 'Н': 1/3, 'Б': 1/3}
            return {k: counter[k]/total for k in ['К', 'Н', 'Б']}
    
    def choose_move(self, strategy='max_ev'):
        probs = self.predict_proba()
        pK, pH, pB = probs['К'], probs['Н'], probs['Б']
        ev = {'К': pH - pB, 'Н': pB - pK, 'Б': pK - pH}
        if strategy == 'max_ev':
            return max(ev, key=ev.get)
        else:
            loss = {'К': pB, 'Н': pK, 'Б': pH}
            return min(loss, key=loss.get)

total = 0
correct = 0
for match_id, group in df.groupby('match_id'):
    predictor = AdaptiveRPSPredictor()
    group = group.sort_values('round')
    for _, row in group.iterrows():
        pred = predictor.choose_move()
        if pred == row['opp_move']:
            correct += 1
        total += 1
        predictor.update(row['opp_move'])

print(f"Марковская модель 1-го порядка: точность {correct}/{total} = {correct/total:.4f}")

NameError: name 'defaultdict' is not defined